# 🦎 Galápagos Wildlife — Banco de Imágenes

Notebook maestro para gestionar las imágenes de entrenamiento compartidas entre:
- `train_galapagos_yolo.ipynb` — detector YOLO con bounding boxes
- `train_galapagos_classifier.ipynb` — clasificador MobileNetV3

**Las imágenes se guardan en Drive y persisten entre sesiones.**
Ejecuta este notebook cada vez que quieras agregar más imágenes o nuevas especies.

```
MyDrive/galapagos_wildlife/
  raw_images/
    00_Amblyrhynchus_cristatus/   ← 500 imgs
    01_Conolophus_subcristatus/   ← 500 imgs
    ...
  manifest.json                  ← registro de descargas por especie
```

## Celda 0 — Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/galapagos_wildlife'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive montado: {DRIVE_ROOT}')


## Celda M — Migración (solo una vez)

Si ya tienes imágenes descargadas en sesiones anteriores, esta celda las mueve
al banco compartido. **Ejecutar solo una vez.** Detecta automáticamente:
- Imágenes del notebook YOLO anterior: `galapagos_yolo/raw_images/`
- Imágenes del notebook classifier anterior: `data/raw/` (local, probablemente perdidas)


In [ ]:
import shutil
from pathlib import Path

# IMAGES_DIR definido aqui porque Celda 1 aun no ha corrido
IMAGES_DIR = Path(f'{DRIVE_ROOT}/raw_images')
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Fuentes de imagenes anteriores
old_sources = [
    Path('/content/drive/MyDrive/galapagos_yolo/raw_images'),   # YOLO anterior
    Path('/content/dataset/raw_images'),                          # local (si aun existe)
]

total_moved = 0
for old_root in old_sources:
    if not old_root.exists():
        print(f'No encontrado: {old_root}')
        continue
    print(f'Migrando desde: {old_root}')
    for cls_dir in sorted(old_root.iterdir()):
        if not cls_dir.is_dir():
            continue
        dst = IMAGES_DIR / cls_dir.name
        dst.mkdir(parents=True, exist_ok=True)
        moved = 0
        for img in cls_dir.glob('*.jpg'):
            target = dst / img.name
            if not target.exists():
                shutil.copy(img, target)
                moved += 1
        total = len(list(dst.glob('*.jpg')))
        print(f'  {cls_dir.name}: {total} imgs en banco (+{moved} migradas)')
        total_moved += moved

total_banco = sum(
    len(list(d.glob('*.jpg')))
    for d in IMAGES_DIR.iterdir() if d.is_dir()
) if IMAGES_DIR.exists() else 0
print(f'\nMigracion completa: {total_moved} imagenes nuevas')
print(f'Banco total: {total_banco} imgs en {DRIVE_ROOT}/raw_images/')


## Celda 1 — Configuración

Ajusta `MAX_PER_CLASS` para controlar cuántas imágenes por especie.
Para agregar una especie nueva, añade una línea a `SPECIES`.


In [ ]:
import os, json, time, random
import urllib.request, urllib.parse
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

# ─── CONFIGURACIÓN ────────────────────────────────────────────────────────────
MAX_PER_CLASS = 500     # Objetivo de imágenes por especie
                        # Para subespecies raras se descargará lo que haya disponible
# ──────────────────────────────────────────────────────────────────────────────

IMAGES_DIR = Path(f'{DRIVE_ROOT}/raw_images')
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# ─── LISTA MAESTRA DE ESPECIES ────────────────────────────────────────────────
# Formato: (id, scientific_name, common_name_en, common_name_es, inat_taxon_id)
# taxon_id = ID exacto de iNaturalist → descarga precisa para TODAS las especies.
# Si una clase queda con 0 imágenes descargadas, simplemente se omite en entrenamiento.
#
# ⚠️  Notas taxonómicas:
#   - ID 2 (Chelonoidis niger genérico) RETIRADO → subespecies en IDs 38-49
#   - ID 6  Sula sula: taxon_id 3793 (taxon_name matchea el género completo)
#   - ID 14 Diomedea irrorata: taxon_id 4143 (= Phoebastria irrorata en iNat)
#   - ID 21 Nesomimus parvulus: taxon_id 73051 (= Mimus parvulus en iNat)
#   - ID 27 Mola mola: taxon_id 49601 (taxon_name matchea una planta)
#   - IDs 38-49: TODAS las subespecies vivas de Chelonoidis niger en Galápagos
#     (incluidas las de pocas obs — se descargará lo disponible y se empieza)
#   - C. n. microphyes (Darwin Volcano): no encontrado como taxón separado en iNat
#   - C. n. niger (Floreana): extinta ~1850, excluida
#
# IDs 3-37 sin cambio → directorios en Drive siguen válidos.
SPECIES = [
    # ── Reptiles ──────────────────────────────────────────────────────────────
    (0,  'Amblyrhynchus cristatus',     'Marine Iguana',            'Iguana Marina',              35347),
    (1,  'Conolophus subcristatus',     'Galapagos Land Iguana',    'Iguana Terrestre',           35324),
    # ID 2 retirado → subespecies de tortuga en IDs 38-49
    # ── Mamíferos marinos ─────────────────────────────────────────────────────
    (3,  'Zalophus wollebaeki',         'Galapagos Sea Lion',       'Lobo Marino',                41738),
    (4,  'Arctocephalus galapagoensis', 'Galapagos Fur Seal',       'Lobo Peletero',              41747),
    # ── Piqueros ──────────────────────────────────────────────────────────────
    (5,  'Sula nebouxii',               'Blue-footed Booby',        'Piquero Patas Azules',        3786),
    (6,  'Sula sula',                   'Red-footed Booby',         'Piquero Patas Rojas',         3793),
    (7,  'Sula granti',                 'Nazca Booby',              'Piquero de Nazca',           56768),
    (8,  'Sula leucogaster',            'Brown Booby',              'Piquero Pardo',               3797),
    # ── Fragatas ──────────────────────────────────────────────────────────────
    (9,  'Fregata magnificens',         'Magnificent Frigatebird',  'Fragata Magnifica',           4631),
    (10, 'Fregata minor',               'Great Frigatebird',        'Fragata Real',                4636),
    # ── Aves marinas ──────────────────────────────────────────────────────────
    (11, 'Pelecanus occidentalis',      'Brown Pelican',            'Pelicano Pardo',              4328),
    (12, 'Nannopterum harrisi',         'Galapagos Cormorant',      'Cormoran no Volador',      1289603),
    (13, 'Spheniscus mendiculus',       'Galapagos Penguin',        'Pinguino de Galapagos',      3815),
    (14, 'Diomedea irrorata',           'Waved Albatross',          'Albatros de Galapagos',      4143),
    (15, 'Buteo galapagoensis',         'Galapagos Hawk',           'Gavillan de Galapagos',      5190),
    # ── Pinzones de Darwin ────────────────────────────────────────────────────
    (16, 'Geospiza magnirostris',       'Large Ground Finch',       'Pinzon Terrestre Grande',    9477),
    (17, 'Geospiza fortis',             'Medium Ground Finch',      'Pinzon Terrestre Mediano',   9479),
    (18, 'Geospiza scandens',           'Cactus Finch',             'Pinzon del Cactus',          9480),
    (19, 'Camarhynchus pauper',         'Medium Tree Finch',        'Pinzon Arboricola',          9475),
    (20, 'Certhidea olivacea',          'Green Warbler Finch',      'Pinzon Cantor Verde',      200000),
    # ── Aves terrestres ───────────────────────────────────────────────────────
    (21, 'Nesomimus parvulus',          'Galapagos Mockingbird',    'Cucuve de Galapagos',       73051),
    (22, 'Pyrocephalus rubinus',        'Vermilion Flycatcher',     'Brujo',                     16447),
    (23, 'Myiarchus magnirostris',      'Galapagos Flycatcher',     'Papamoscas de Galapagos',  16039),
    (24, 'Zenaida galapagoensis',       'Galapagos Dove',           'Paloma de Galapagos',       3451),
    # ── Fauna marina ──────────────────────────────────────────────────────────
    (25, 'Aetobatus narinari',          'Spotted Eagle Ray',        'Raya Aguila Moteada',      49297),
    (26, 'Triaenodon obesus',           'Whitetip Reef Shark',      'Tiburon Punta Blanca',     52314),
    (27, 'Mola mola',                   'Ocean Sunfish',            'Pez Luna',                  49601),
    (28, 'Chelonia mydas',              'Green Sea Turtle',         'Tortuga Marina Verde',     39659),
    (29, 'Balaenoptera musculus',       'Blue Whale',               'Ballena Azul',             41553),
    (30, 'Tursiops truncatus',          'Bottlenose Dolphin',       'Delfin Nariz de Botella',  41482),
    # ── Aves costeras / crustáceos ────────────────────────────────────────────
    (31, 'Haematopus palliatus',        'American Oystercatcher',   'Ostrero Americano',         4840),
    (32, 'Grapsus grapsus',             'Sally Lightfoot Crab',     'Cangrejo Zayapa',          55295),
    (33, 'Egretta thula',               'Snowy Egret',              'Garza Blanca',              4940),
    (34, 'Nycticorax nycticorax',       'Black-crowned Night Heron','Garza Nocturna',            4981),
    (35, 'Puffinus subalaris',          'Galapagos Shearwater',     'Petrel de Galapagos',     144440),
    (36, 'Anous stolidus',              'Brown Noddy',              'Gaviotin Cafe',             4535),
    (37, 'Creagrus furcatus',           'Swallow-tailed Gull',      'Gaviota Cola Bifurcada',   4564),
    # ── Tortugas Gigantes — TODAS las subespecies conocidas en Galápagos ──────
    # Las de pocas obs (hoodensis, becki, darwini, etc.) descargarán lo disponible.
    # Si quedan con 0 imágenes se omiten solas en la anotación y entrenamiento.
    # Con el tiempo y más reportes en iNaturalist estas clases irán creciendo.
    # C. n. microphyes (Volcán Darwin) no aparece como taxón separado en iNat.
    # C. n. niger (Floreana, extinta ~1850) excluida.
    (38, 'Chelonoidis niger porteri',      'Santa Cruz Dome Tortoise',    'Tortuga Cupula Santa Cruz', 1367386),  # 1457 RG
    (39, 'Chelonoidis niger donfaustoi',   'Santa Cruz East Tortoise',    'Tortuga Este Santa Cruz',   1367393),  #  115 RG
    (40, 'Chelonoidis niger vandenburghi', 'Alcedo Volcano Tortoise',     'Tortuga Volcan Alcedo',     1367385),  #  104 RG
    (41, 'Chelonoidis niger guntheri',     'Sierra Negra Tortoise',       'Tortuga Sierra Negra',      1367391),  #  242 RG
    (42, 'Chelonoidis niger chathamensis', 'San Cristobal Tortoise',      'Tortuga San Cristobal',     1367395),  #  132 RG
    (43, 'Chelonoidis niger hoodensis',    'Espanola Saddleback Tortoise','Tortuga Sillin Espanola',   1367390),  #    6 RG ← pocas, se descarga todo
    (44, 'Chelonoidis niger vicina',       'Cerro Azul Tortoise',         'Tortuga Cerro Azul',        1367384),  #    4 RG
    (45, 'Chelonoidis niger darwini',      'Santiago Tortoise',           'Tortuga de Santiago',       1367394),  #    2 RG
    (46, 'Chelonoidis niger duncanensis',  'Pinzon Tortoise',             'Tortuga de Pinzon',         1367392),  #    1 RG
    (47, 'Chelonoidis niger becki',        'Wolf Volcano Tortoise',       'Tortuga Volcan Wolf',       1367396),  #    2 RG
    (48, 'Chelonoidis niger abingdonii',   'Pinta Tortoise',              'Tortuga de Pinta',          1367404),  #    1 RG (Lonesome George)
    (49, 'Chelonoidis niger phantasticus', 'Fernandina Tortoise',         'Tortuga de Fernandina',     1367387),  #    0 RG (casi extinta)
    # ── Agregar nuevas especies aquí ──────────────────────────────────────────
    # (50, 'Scientific name', 'Common EN', 'Comun ES', inat_taxon_id),
]

# Guardar en Drive como fuente única de verdad
config = {
    'max_per_class': MAX_PER_CLASS,
    'species': [
        {'id': sid, 'scientific': sci, 'en': en, 'es': es, 'taxon_id': tid}
        for sid, sci, en, es, tid in SPECIES
    ]
}
config_path = Path(f'{DRIVE_ROOT}/species_config.json')
config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False))

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

manifest_path = Path(f'{DRIVE_ROOT}/manifest.json')
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

total_existing = 0
print(f'\n{"ID":>4}  {"Especie":<46} {"Imgs":>6}  {"Estado":>14}')
print('-' * 76)
for sid, sci, en, es, tid in SPECIES:
    cls_key = f"{sid:02d}_{sci.replace(' ', '_')}"
    cls_dir = IMAGES_DIR / cls_key
    n = len(list(cls_dir.glob('*.jpg'))) if cls_dir.exists() else 0
    total_existing += n
    status = 'OK' if n >= MAX_PER_CLASS else (f'{n} imgs' if n > 0 else 'pendiente')
    print(f'{sid:>4}  {en:<46} {n:>6}   {status:<14}  [tid:{tid}]')

need = max(0, MAX_PER_CLASS * len(SPECIES) - total_existing)
print(f'\nTotal: {total_existing} imgs  |  {len(SPECIES)} especies (todas con taxon_id)')
print(f'species_config.json guardado en Drive')

## Celda 2 — Descarga de Imágenes

In [ ]:
def fetch_obs_paginated(sci_name, max_results, place_id=None, taxon_id=None):
    """Busca observaciones en iNaturalist.
    Si taxon_id está definido, usa búsqueda por ID (más precisa — recomendado
    para grupos con taxonomía compleja como tortugas gigantes).
    """
    all_results = []
    page = 1
    while len(all_results) < max_results:
        params = {
            'quality_grade': 'research',
            'photos': 'true', 'per_page': 200, 'page': page, 'order_by': 'votes',
        }
        if taxon_id:
            params['taxon_id'] = taxon_id   # más preciso que taxon_name
        else:
            params['taxon_name'] = sci_name
        if place_id:
            params['place_id'] = place_id
        url = 'https://api.inaturalist.org/v1/observations?' + urllib.parse.urlencode(params)
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                batch = json.loads(r.read()).get('results', [])
                if not batch:
                    break
                all_results.extend(batch)
                if len(batch) < 200:
                    break
                page += 1
                time.sleep(0.5)
        except Exception:
            break
    return all_results[:max_results]

def collect_urls(results):
    seen, urls = set(), []
    for obs in results:
        for photo in obs.get('photos', []):
            url = photo.get('url', '').replace('square', 'medium')
            if url and url not in seen:
                seen.add(url)
                urls.append(url)
    return urls

n_species = len(SPECIES)
log(f'=== DESCARGA iNaturalist — {n_species} especies x {MAX_PER_CLASS} imgs ===')

for class_id, sci_name, en_name, es_name, inat_taxon_id in SPECIES:
    cls_key = f"{class_id:02d}_{sci_name.replace(' ', '_')}"
    cls_dir = IMAGES_DIR / cls_key
    cls_dir.mkdir(exist_ok=True)
    existing = list(cls_dir.glob('*.jpg'))
    prefix = f'[{class_id:02d}]'

    if len(existing) >= MAX_PER_CLASS:
        print(f'{prefix} ✓ {en_name:<40} {len(existing)} imgs (skip)')
        manifest[cls_key] = len(existing)
        continue

    tid_info = f'taxon_id={inat_taxon_id}' if inat_taxon_id else f'name="{sci_name}"'
    galapagos = collect_urls(fetch_obs_paginated(sci_name, MAX_PER_CLASS * 3, 7161, inat_taxon_id))
    global_r  = collect_urls(fetch_obs_paginated(sci_name, MAX_PER_CLASS * 3, None, inat_taxon_id))
    seen_g    = set(galapagos)
    all_urls  = galapagos + [u for u in global_r if u not in seen_g]

    downloaded = len(existing)
    print(f'{prefix} ⬇ {en_name:<40} {len(galapagos)} Galápagos + {len(global_r)} global [{tid_info}]')

    with tqdm(total=MAX_PER_CLASS, initial=downloaded,
              desc=f'     {en_name[:28]}', unit='img',
              bar_format='{l_bar}{bar:20}{r_bar}') as pbar:
        for url in all_urls:
            if downloaded >= MAX_PER_CLASS:
                break
            fname = cls_dir / f'{downloaded:04d}.jpg'
            if fname.exists():
                downloaded += 1
                pbar.update(1)
                continue
            try:
                urllib.request.urlretrieve(url, fname)
                downloaded += 1
                pbar.update(1)
            except Exception:
                pass

    manifest[cls_key] = downloaded
    manifest_path.write_text(json.dumps(manifest, indent=2))
    log(f'{prefix} ✓ {en_name}: {downloaded} imgs — manifest actualizado')
    time.sleep(0.3)

total = sum(manifest.values())
log(f'=== DESCARGA COMPLETA: {total} imgs en {len(manifest)} especies ===')

## Celda 2b — Descarga SOLO tortugas gigantes (12 subespecies, IDs 38-49)

Ejecuta esta celda si ya tienes las otras 37 especies descargadas y solo necesitas
las subespecies de tortuga. Mucho más rápido que ejecutar Celda 2 completa.
Las subespecies raras (< 10 obs) descargarán lo que haya disponible.

In [ ]:
print(f'{"ID":>4}  {"Especie":<40} {"Imgs":>6} {"Objetivo":>8} {"Estado":>10}')
print('-' * 74)
total = 0
completas = 0
for class_id, sci_name, en_name, es_name, inat_taxon_id in SPECIES:
    cls_key = f"{class_id:02d}_{sci_name.replace(' ', '_')}"
    cls_dir = IMAGES_DIR / cls_key
    n = len(list(cls_dir.glob('*.jpg'))) if cls_dir.exists() else 0
    total += n
    ok = n >= MAX_PER_CLASS
    if ok:
        completas += 1
    bar = '#' * int(n / MAX_PER_CLASS * 20)
    status = 'OK' if ok else f'{n}/{MAX_PER_CLASS}'
    print(f'{class_id:>4}  {en_name:<40} {n:>6} {MAX_PER_CLASS:>8}   {status:<10}  [{bar:<20}]')

print('-' * 74)
print(f'{"TOTAL":<46} {total:>6}   {completas}/{len(SPECIES)} especies completas')
print(f'\nRuta: {IMAGES_DIR}')
print(f'\nNota: ID 2 (Chelonoidis niger) retirado. Tortugas específicas: IDs 38-43.')

## Celda 3 — Estadísticas del banco

In [ ]:
print(f'{"Especie":<40} {"Imgs":>6} {"Objetivo":>8} {"Estado":>10}')
print('-' * 68)
total = 0
completas = 0
for class_id, sci_name, en_name, es_name, inat_taxon_id in SPECIES:
    cls_key = f"{class_id:02d}_{sci_name.replace(' ', '_')}"
    cls_dir = IMAGES_DIR / cls_key
    n = len(list(cls_dir.glob('*.jpg'))) if cls_dir.exists() else 0
    total += n
    ok = n >= MAX_PER_CLASS
    if ok:
        completas += 1
    bar = '#' * int(n / MAX_PER_CLASS * 20)
    status = 'OK' if ok else f'{n}/{MAX_PER_CLASS}'
    print(f'{en_name:<40} {n:>6} {MAX_PER_CLASS:>8}   {status:<10}  [{bar:<20}]')

print('-' * 68)
print(f'{"TOTAL":<40} {total:>6}   {completas}/{len(SPECIES)} especies completas')
print(f'\nRuta: {IMAGES_DIR}')
